In [1]:
# Network segments to evaluate. Edit these before running.
# Enter any address in the target /24 (e.g., "10.1.5.0" or "10.1.5.0/24").
# Only the first three octets are used.
oco = "143.88.1.0"   # OCO: the subnet YOUR group attacked
dco = "143.88.10.0"   # DCO: YOUR group's subnet, attacked by another group


In [2]:
# -*- coding: utf-8 -*-
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
import traceback
import sys
import datetime
import pandas as pd
import warnings
warnings.filterwarnings("ignore")   # hide Python warnings; errors still show
from IPython.display import display

from df_binning import *
from generate_individual_labels_dataframe import *
import pyspark.sql.functions as F
from pyspark.sql.window import Window

#############
# Run variables
#############

benign_label = "none"

# TEST: read the combined class dataset directly from HDFS
conn_server_loc = 'hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv'

# TEST: log on this machine (normal file writes cannot go to HDFS)
log_location = "/home/ubuntu/rf_logs/" + str(datetime.date.today()) + "_log.txt"
import os
os.makedirs(os.path.dirname(log_location), exist_ok=True)   # create the log folder if it is missing

########################
# functions
########################

def segment_prefix(addr):
    """First three octets plus a trailing dot, e.g. '10.1.5.0/24' -> '10.1.5.'"""
    octets = addr.strip().split("/")[0].split(".")
    if len(octets) < 3 or not all(o.isdigit() and 0 <= int(o) <= 255 for o in octets[:3]):
        raise ValueError("'" + addr + "' is not a valid IPv4 network segment")
    return ".".join(octets[:3]) + "."

GAP_SECONDS = 120   # same 2-minute window the correlation script uses

def segment_attack_events(preds, ip_col, prefix, gap_seconds=GAP_SECONDS):
    """Condenses a segment's attack connections into attacks so counts can be compared with
    mission logs. Connections from the same source IP with the same technique (T code) belong
    to one attack until there is a gap of more than gap_seconds between them; a longer gap
    starts a new attack. Rows the correlation script marked "Duplicate" (extra copies made
    when one technique maps to several tactics) are skipped so they are not counted twice.
    An attack counts as detected if the model flagged at least one of its connections.
    Returns (number of attacks, number of detected attacks, table with one row per attack)."""
    attacks = (preds
               .filter(F.col(ip_col).cast("string").startswith(prefix)
                       & (F.col("label_bin") == 0.0)
                       & (F.coalesce(F.col("label_technique").cast("string"), F.lit("")) != "Duplicate")
                       & F.col("ts").isNotNull())
               .select(F.col("src_ip").cast("string").alias("src"),
                       F.col("label_technique").cast("string").alias("technique"),
                       F.col("ts").cast("double").alias("ts_s"), "prediction"))
    w = Window.partitionBy("src", "technique").orderBy("ts_s")
    attacks = attacks.withColumn("gap", F.col("ts_s") - F.lag("ts_s").over(w))
    attacks = attacks.withColumn("new_attack",
                                 F.when(F.col("gap").isNull() | (F.col("gap") > gap_seconds), 1).otherwise(0))
    attacks = attacks.withColumn("attack_id",
                                 F.sum("new_attack").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    per_attack = (attacks.groupBy("src", "technique", "attack_id")
                  .agg(F.from_unixtime(F.min("ts_s").cast("long")).alias("Start (UTC)"),
                       F.from_unixtime(F.max("ts_s").cast("long")).alias("End (UTC)"),
                       F.count(F.lit(1)).alias("Connections"),
                       F.sum(F.when(F.col("prediction") == 0.0, 1).otherwise(0)).alias("Flagged"))
                  .orderBy("Start (UTC)")
                  .toPandas())
    # Detection rule: an attack counts as detected if the model flagged at least one of its connections.
    detected = per_attack["Flagged"] > 0
    # Stricter rule (uncomment to use): at least 70% of the attack's connections must be flagged.
    detected = per_attack["Flagged"] >= 0.7 * per_attack["Connections"]
    per_attack["Flagged %"] = (100 * per_attack["Flagged"] / per_attack["Connections"]).round(0).astype(int)
    per_attack["Detected"] = detected.map({True: "Yes", False: "No"})
    per_attack = per_attack.rename(columns={"technique": "T code", "src": "Source IP"})
    per_attack = per_attack[["T code", "Source IP", "Start (UTC)", "End (UTC)",
                             "Connections", "Flagged", "Flagged %", "Detected"]]
    per_attack.index = range(1, len(per_attack) + 1)
    n = len(per_attack)
    d = int(detected.sum())
    return n, d, per_attack

def segment_benign_events(preds, ip_col, prefix, gap_seconds=GAP_SECONDS):
    """Groups a segment's benign connections (no T code, label "none") the same way as attacks,
    minus the T code: connections from the same source IP belong to one benign event until
    there is a gap of more than gap_seconds between them. An event counts as a false alarm
    if the model flagged at least one of its connections as an attack.
    Returns (number of benign events, number of flagged events)."""
    benign = (preds
              .filter(F.col(ip_col).cast("string").startswith(prefix)
                      & (F.col("label_bin") == 1.0)
                      & F.col("ts").isNotNull())
              .select(F.col("src_ip").cast("string").alias("src"),
                      F.col("ts").cast("double").alias("ts_s"), "prediction"))
    w = Window.partitionBy("src").orderBy("ts_s")
    benign = benign.withColumn("gap", F.col("ts_s") - F.lag("ts_s").over(w))
    benign = benign.withColumn("new_event",
                               F.when(F.col("gap").isNull() | (F.col("gap") > gap_seconds), 1).otherwise(0))
    benign = benign.withColumn("event_id",
                               F.sum("new_event").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    per_event = benign.groupBy("src", "event_id").agg(
        F.max(F.when(F.col("prediction") == 0.0, 1).otherwise(0)).alias("flagged"))
    r = per_event.agg(F.count(F.lit(1)).alias("n"), F.sum("flagged").alias("f")).collect()[0]
    return int(r["n"]), int(r["f"] or 0)

oco_prefix = segment_prefix(oco)
dco_prefix = segment_prefix(dco)
print("OCO segment (dest_ip): " + oco_prefix + "x")
print("DCO segment (dest_ip): " + dco_prefix + "x")

########################
# code start
########################

spark = ( 
    SparkSession
    .builder
    .master("local[*]")
    .appName("TM_RandoForest")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
    )

# Only show Spark errors (hides WARN/INFO messages such as maxToStringFields)
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")   # times shown in UTC, matching the dataset

printToLog("Begin run\n-----------\n----------\n\n", log_location)
printToLog("Dataset location - " + conn_server_loc, log_location)

source_df = spark.read.option("header", True).options(inferSchema=True).csv(conn_server_loc)
 
try:
    source_df = rename_columns(source_df)
except Exception as e:
    print("Error:  " + str(e), file=sys.stderr)
    print(traceback.format_exc())   

# Binary label: any technique label other than "none" = attack (0.0), "none" = benign (1.0)
unbinned_df = source_df.withColumn("label_bin", when((col("label_multi") != benign_label), 0.0).otherwise(1.0))
printToLog(str(unbinned_df.count()) + " rows", log_location)

#Carry out the binning on the df
numeric_percent_trim = 0.02
nominal_percent_agg = 0.98
replace_bool = False

attrList_ip_addr = ['dest_ip','src_ip']
attrList_port = ['dest_port', 'src_port']
attrList_bool = ['local_orig', 'local_resp']
attrList_nominal = ['protocol', 'conn_state', 'history','service']
attrList_numeric = ['duration','orig_bytes','orig_pkts','orig_ip_bytes','resp_bytes','resp_pkts','resp_ip_bytes','missed_bytes']

printToLog("Binning start", log_location)
conn_df = genFullBinnedDF(unbinned_df, attrList_ip_addr, attrList_port, attrList_bool, attrList_nominal, nominal_percent_agg, attrList_numeric, numeric_percent_trim, replace_bool).persist()
printToLog("Binning finished", log_location)

# balanced class weights rarer classes get a higher weight so the
# model is penalized more for missing them. Same formula sklearn uses for
# class_weight='balanced': total / (num_classes * class_count).
class_counts = conn_df.groupBy("label_bin").count()
total_count = conn_df.count()
num_classes = class_counts.count()
class_counts = class_counts.withColumn("class_weight", total_count / (num_classes * col("count")))
conn_df = conn_df.join(class_counts.select("label_bin", "class_weight"), on="label_bin", how="left").persist()

########################
# Build Feature cols
########################     

feature_cols = []
for clm, types in conn_df.dtypes:
    if "_bin" in clm:
        if not "label" in clm:
            feature_cols.append(clm)

# build feature vector for Classifier
assembler = VectorAssembler(inputCols = feature_cols, outputCol = "features")
conn_df = assembler.transform(conn_df)

printToLog("feature_cols - " + str(feature_cols), log_location)
print(str(len(feature_cols)) + " features: " + str(feature_cols))

########################
# Model: trained on every connection EXCEPT those going to the OCO and DCO subnets,
# so the model is scored on traffic it has never seen
########################

in_segments = coalesce(col("dest_ip").cast("string").startswith(oco_prefix) |
                       col("dest_ip").cast("string").startswith(dco_prefix), lit(False))

# Also leave out all traffic FROM the subnets that attacked the OCO/DCO subnets,
# so the model has not seen those attackers' activity anywhere else either
attacker_prefixes = [r["p"] for r in
                     conn_df.filter(in_segments & (col("label_bin") == 0.0))
                            .select(F.concat(F.substring_index(col("src_ip").cast("string"), ".", 3),
                                             F.lit(".")).alias("p"))
                            .distinct().collect()]
from_attackers = lit(False)
for p in attacker_prefixes:
    from_attackers = from_attackers | col("src_ip").cast("string").startswith(p)

train_df = conn_df.filter(~in_segments & ~coalesce(from_attackers, lit(False)))
score_df = conn_df.filter(in_segments)
print("Left out of training: traffic to", oco_prefix + "x,", dco_prefix + "x and from", attacker_prefixes)

rf = RandomForestClassifier(featuresCol = "features", labelCol = "label_bin",
                            weightCol = "class_weight", seed = 2057)
rfModel = rf.fit(train_df)
printToLog("randomForest model fit", log_location)

#############
# Predictions on the OCO and DCO connections only
#############   

predictions = rfModel.transform(score_df)

for ip_c in ("dest_ip", "src_ip", "ts", "label_technique", "label_multi"):
    if ip_c not in predictions.columns:
        raise RuntimeError("'" + ip_c + "' is not in the dataframe; the OCO/DCO counts need it")

################################
# Output: confusion matrices counting only connections destined to the OCO and DCO subnets
################################

segments = [("OCO", "dest_ip", oco_prefix), ("DCO", "dest_ip", dco_prefix)]

# Event-level confusion matrix, same table format as TM_ML.randForestMaster.
# Positive = attack. Connections are grouped so counts can be compared with mission logs:
#   TP / FN: attacks (same source IP and T code, split at gaps over 2 minutes) detected / missed
#   FP / TN: benign events (same source IP, split at gaps over 2 minutes) flagged / not flagged
for seg_name, ip_col, prefix in segments:
    n_conn = predictions.filter(col(ip_col).cast("string").startswith(prefix)).count()
    n_events, tp, attack_table = segment_attack_events(predictions, ip_col, prefix)
    n_windows, fp = segment_benign_events(predictions, ip_col, prefix)
    fn = n_events - tp
    tn = n_windows - fp

    two_by_two = pd.DataFrame(
        [[f"TP: {tp}", f"FN: {fn}"],
         [f"FP: {fp}", f"TN: {tn}"]],
        index=["Actual: Attack", "Actual: Benign"],
        columns=["Predicted: Attack", "Predicted: Benign"]
    )

    print(f"\n{seg_name}:")
    print(f"  {ip_col} {prefix}x  |  {n_conn:,} connections")
    print(f"  Attacks: {n_events:,}")
    print(f"  Benign:  {n_windows:,}")
    display(two_by_two)

    # Same formulas as TM_ML.randForestMaster, with attack as the positive class
    total = tp + fn + fp + tn
    accuracy = (tp + tn) / total if total > 0 else float("nan")
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else float("nan"))
    print(f"  Accuracy: {accuracy:.1%}   "
          f"Precision: {precision:.1%}   "
          f"Recall: {recall:.1%}   "
          f"F1: {f1:.1%}")

    # Connection-level view: how much of the attack traffic itself the model recognized
    attack_conns = int(attack_table["Connections"].sum()) if n_events else 0
    flagged_conns = int(attack_table["Flagged"].sum()) if n_events else 0
    if attack_conns:
        print(f"  Attack connections flagged: {flagged_conns:,} of {attack_conns:,} "
              f"({flagged_conns / attack_conns:.0%})")
    if n_conn == 0:
        print(f"  No connections matched {prefix}x - check the {seg_name} address.")

    # One row per counted attack, for matching against mission log entries
    print("  Attacks counted:")
    if n_events == 0:
        print("  (no connections to this subnet are labeled as attacks in the dataset)")
    else:
        display(attack_table)

conn_df.unpersist()
printToLog("End run\n-----------\n----------\n\n", log_location)


OCO segment (dest_ip): 143.88.1.x
DCO segment (dest_ip): 143.88.10.x


26/10/27 17:44:47 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.131 instead (on interface ens33)
26/10/27 17:44:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 17:44:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/27 17:44:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

18 features: ['dest_ip_bin', 'src_ip_bin', 'dest_port_bin', 'src_port_bin', 'local_orig_bin', 'local_resp_bin', 'protocol_bin', 'conn_state_bin', 'history_bin', 'service_bin', 'duration_bin', 'orig_bytes_bin', 'orig_pkts_bin', 'orig_ip_bytes_bin', 'resp_bytes_bin', 'resp_pkts_bin', 'resp_ip_bytes_bin', 'missed_bytes_bin']


NameError: name 'in_segments' is not defined